# Notebook 05 — Stage 2: two-class $k$-shortest-path assignment

*Thesis §Assignment, §Low-Stress Link Set.*

Both rider classes route on the full network $N_{full}$; they differ only in the edge cost that ranks
routes:

| class | cost | source |
|---|---|---|
| stress-tolerant | $c^{high}_e=\tau_e$ (grade-adjusted time) | Parkin & Rotheram (2010) |
| low-stress | $c^{low}_e=\lambda_{stress}\,\tau_e$ if LTS$_e>1$, else $\tau_e$ | Broach et al. (2012), $\lambda=2.4$ |

For every O–D pair the $k$ shortest loopless paths (Yen 1971) are generated and demand is split evenly
across them. `K_STORE = 5` paths are stored per pair so Notebooks 07 and 09b can assemble $k=1\dots5$
**without re-routing** (the first 3 of 5 Yen paths are exactly the $k=3$ set).

**No count leakage.** Routing uses only the network and the O–D node pairs. Every count-based figure in
this notebook is a *training-station* diagnostic at the seed share $\pi^{(0)}$.

**Inputs** (Notebook 04): `network_lts_grade.graphml`, `od_calibrated_t.csv`, `A_neutral.npz`,
`A_neutral_links.csv`, `count_link_members.csv`, `train_mask.npy`, `zone_rep_nodes.json`.
**Outputs:** `A_low.npz`, `A_high.npz`, `A_composite_pi0.npz`, `stage2_link_fit.csv`,
`lowstress_route_stats.csv`, cached path stores in `data/processed/paths/`, and the Stage-2 text report.

**Runtime.** Routing is the slow step (~1–3 s per pair per class). Set `N_JOBS` to the number of CPU
cores; results are identical to a serial run and are cached, so a re-run reuses the stored paths.


## Setup and configuration

Imports, shared thesis style, and the parallel-routing job count. `N_JOBS` only affects speed — the
cached path stores make every run deterministic.


In [2]:
import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy import sparse
import thesis_common as tc

tc.set_style()

# Use all CPU cores but one for parallel routing; set to 1 to route serially (identical result).
N_JOBS = max(1, (os.cpu_count() or 2) - 1)
NB = "05"

print("Run stamp:", tc.RUN_STAMP, "| N_JOBS:", N_JOBS)
display(tc.config_table())


Run stamp: 20261104_2223 | N_JOBS: 9


,parameter,value,source
0,v_flat (km/h),21.6,Parkin & Rotheram (2010)
1,kappa_up (km/h per %),1.44,Parkin & Rotheram (2010)
2,kappa_down (km/h per %),0.86,Parkin & Rotheram (2010)
3,v_min (km/h),6.0,Parkin & Rotheram (2010)
4,E_low threshold (LTS <=),1,Mekuria et al. (2012); conservative choice
5,lambda_stress,2.4,"Broach et al. (2012) commute, 20-30k AADT (+140%)"
6,k paths,3,modelling choice; swept 1-5 (NB09b)
7,pi_low0,0.81,Dill & McNeil (2013): 56/69
8,pi grid / tol / max outer,101 / 0.01 / 25,method choice
9,sigma0^2,10.0,no literature value; train-only CV check in NB09a


# A — Network and the two class costs


Load the routable graph, stamp the two class costs onto it ($c^{high}$ = grade-adjusted time;
$c^{low}$ = that time times $\lambda_{stress}$ on links outside the low-stress set), report the size of
the low-stress link set $E_{low}$, and collapse the multigraph to the two simple routing graphs.


In [3]:
# Load the routable network built in Notebook 02.
Gr = tc.load_network()

# Stamp both class costs: baseline stress penalty (lam=2.4), no additive climb penalty, grade-adjusted speed.
tc.set_costs(Gr, lam=tc.STRESS_PENALTY, beta=0.0, flat=False)

n_e = Gr.number_of_edges()

# Size of the low-stress link set E_low (LTS <= threshold), by edge count and by length per LTS tier.
low_e = sum(1 for *_, d in Gr.edges(keys=True, data=True) if d["lts"] <= tc.LOW_STRESS_MAX_LTS)
km = pd.Series({l: sum(d["len_m"] for *_, d in Gr.edges(keys=True, data=True) if d["lts"] == l) / 1000
                for l in [1, 2, 3, 4]})

print(f"G_scc: {Gr.number_of_nodes():,} nodes, {n_e:,} directed edges")
print(f"E_low (LTS<={tc.LOW_STRESS_MAX_LTS}): {low_e:,} edges ({100*low_e/n_e:.0f}%), "
      f"{km[1]:.0f} of {km.sum():.0f} km ({100*km[1]/km.sum():.0f}% of length)")

# One simple DiGraph per class (lowest-cost parallel edge kept), used for the k-shortest routing.
Gs_high = tc.to_simple(Gr, "cost_high")
Gs_low  = tc.to_simple(Gr, "cost_low")
print(f"simple graphs: {Gs_high.number_of_edges():,} edges each")


G_scc: 21,212 nodes, 44,411 directed edges
E_low (LTS<=1): 17,905 edges (40%), 797 of 1944 km (41% of length)
simple graphs: 44,257 edges each


# B — Demand, monitored links (corridor members), training stations


Load the calibrated demand, the monitored-link metadata and their corridor member edges, the training
mask, and the neutral matrix. Each zone-level O–D pair is then expanded over its sampled representative
nodes (the zone-spread sub-pairs that Stage 2 actually routes).


In [4]:
# Calibrated demand and monitored-link metadata from Notebook 04.
cal  = pd.read_csv(tc.DATA_PROCESSED / "od_calibrated_t.csv").sort_values("pair_idx").reset_index(drop=True)
links = pd.read_csv(tc.DATA_PROCESSED / "A_neutral_links.csv").sort_values("link_idx").reset_index(drop=True)
L, P  = len(links), len(cal)
v_obs = links["v_obs"].to_numpy(float)
t_cal = cal["t"].to_numpy(float); n_seed = cal["n_seed"].to_numpy(float)

# Corridor members map each monitored station to all directed edges of its cross-section (the screenline).
mon   = tc.load_members(L)
assert set(mon.values()) == set(range(L)), "every monitored link needs >=1 member edge (re-run NB04)"

train = tc.load_train_mask(L)
A_neutral = sparse.load_npz(tc.DATA_PROCESSED / "A_neutral.npz").tocsr()
pairs = list(zip(cal["o_node"], cal["d_node"]))

# Zone-spread expansion: each O-D pair -> up to N_REP x N_REP non-degenerate node sub-pairs to route.
exp_pairs, assembly, rep_idx = tc.expand_pairs(cal)

print(f"zone-spread: {P:,} O-D pairs -> {len(exp_pairs):,} unique sub-pairs (N_REP={tc.N_REP}); "
      f"{sum(1 for a in assembly if not a)} pairs with no routable sub-pair")
print(f"{P:,} O-D pairs (sum t {t_cal.sum():,.0f}) | {L} monitored links -> {len(mon):,} member edges")
print(f"training stations: {int(train.sum())}  (held-out stations are not loaded here)")

# Flag any intra-node pairs (origin node == destination node): they get no path and carry no Stage-2 flow.
same = int(sum(1 for o, d in pairs if o == d))
if same:
    print(f"  ! {same} pairs have identical origin/destination nodes (intra-node trips): they get no path "
          f"and carry no flow in Stage 2 - check NB04 Section E if this is more than a handful")


zone-spread: 1,850 O-D pairs -> 8,721 unique sub-pairs (N_REP=5); 0 pairs with no routable sub-pair
1,850 O-D pairs (sum t 3,714) | 77 monitored links -> 290 member edges
training stations: 53  (held-out stations are not loaded here)


# C — $k$-shortest path stores and the class incidence matrices


Run Yen's $k$-shortest-path algorithm (in parallel) for every sub-pair in each class, assemble the sparse
class incidence matrices $A_{low}$ and $A_{high}$, and build the baseline composite $A(\pi^{(0)})$. Both
stores are cached to disk, so a re-run loads them instead of re-routing.


In [5]:
# Route (or load cached) K_STORE paths per sub-pair for each class. This is the slow step on a cold cache.
t0 = time.time()
store_high = tc.route_store(Gs_high, exp_pairs, tc.K_STORE, "cost_high", tc.store_tag("high"), n_jobs=N_JOBS)
store_low  = tc.route_store(Gs_low,  exp_pairs, tc.K_STORE, "cost_low",  tc.store_tag("low"),  n_jobs=N_JOBS)
print(f"routing wall time {(time.time()-t0)/60:.1f} min")

# Assemble the class incidence matrices at the baseline k, then the composite at the seed share pi_0.
A_high = tc.assemble_A_spread(store_high, tc.K_PATHS, mon, L, assembly, P)
A_low  = tc.assemble_A_spread(store_low,  tc.K_PATHS, mon, L, assembly, P)
A_pi0  = tc.composite(A_low, A_high, tc.PI_LOW0)

# Report sub-pairs that returned no path or fewer than k paths (expected to be a tiny handful).
for nm, S in [("high", store_high), ("low", store_low)]:
    npth = np.array([len(s) for s in S])
    print(f"  {nm:<4}: sub-pairs with 0 paths {int((npth==0).sum())}, with <{tc.K_PATHS} paths "
          f"{int(((npth>0)&(npth<tc.K_PATHS)).sum())}")
print(f"  O-D pairs with no routable sub-pair (carry no Stage-2 flow): {sum(1 for a in assembly if not a)}")
print(f"A_high nnz {A_high.nnz:,} | A_low nnz {A_low.nnz:,}")


  [high_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
  [low_lam2.4_beta0_grade_k5_sprd5] loaded cached path store (8,721 pairs, k=5)
routing wall time 0.0 min
  high: sub-pairs with 0 paths 0, with <3 paths 6
  low : sub-pairs with 0 paths 0, with <3 paths 6
  O-D pairs with no routable sub-pair (carry no Stage-2 flow): 0
A_high nnz 3,907 | A_low nnz 4,959


# D — Diagnostics


Four training-only checks that justify the method: **D1** identifiability (do the two classes load
different monitored links, so $\pi_{low}$ can be estimated), **D2** $k$-path diversity, **D3** how close
the zone-spread representation sits to the $m\times n$ neutral assignment, and **D4** the training fit at
the seed share $\pi^{(0)}$.


In [7]:
# D1 — identifiability: the two classes must load monitored links differently, or pi_low is not estimable.
adiff = tc.a_diff(A_low, A_high)
row = np.asarray(abs(A_low - A_high).sum(axis=1)).ravel()                  # per-link low/high difference
f_low, f_high = np.asarray(A_low.dot(t_cal)).ravel(), np.asarray(A_high.dot(t_cal)).ravel()
print(f"D1 |A_low-A_high|_1/|A_high|_1 = {adiff:.0%}  "
      f"({'distinct -> pi_low identifiable' if adiff > 0.05 else 'near-equal -> pi_low weakly identified'})")
print(f"   monitored links with a low/high difference: {int((row>1e-9).sum())}/{L}")
print(f"   flow lever |f(pi=1)-f(pi=0)|/sum f(pi=0) = {np.abs(f_low-f_high).sum()/max(f_high.sum(),1e-9):.0%}")

# D2 — path-set diversity: how different are a pair's k paths (edge overlap and cost spread)?
def _jacc(p, q):
    """Jaccard overlap of the directed-edge sets of two node paths."""
    a = set(zip(p[:-1], p[1:])); b = set(zip(q[:-1], q[1:]))
    return len(a & b) / max(len(a | b), 1)
div = []
for cls, S, G in [("high", store_high, Gs_high), ("low", store_low, Gs_low)]:
    w = "cost_high" if cls == "high" else "cost_low"
    for pid, ps in enumerate(S):
        ps = ps[:tc.K_PATHS]
        if len(ps) < 2: continue
        c = [sum(G[int(a)][int(b)][w] for a, b in zip(p[:-1], p[1:])) for p in ps]
        ov = np.mean([_jacc(ps[i], ps[j]) for i in range(len(ps)) for j in range(i+1, len(ps))])
        div.append(dict(cls=cls, pid=pid, overlap=ov, cost_spread=c[-1]/c[0]-1 if c[0] > 0 else 0))
div = pd.DataFrame(div)
for cls, g in div.groupby("cls"):
    print(f"D2 {cls:<4}: median edge overlap among k={tc.K_PATHS} paths {g.overlap.median():.0%}, "
          f"median cost of path {tc.K_PATHS} vs path 1 +{100*g.cost_spread.median():.1f}%")

# D3 — O-D representation: compare AON (k=1) incidence under three spatial representations against the
# m x n neutral matrix from NB04 (single rep node vs N_REP zone-spread).
A_spread_aon = tc.assemble_A_spread(store_high, 1, mon, L, assembly, P)                         # N_REP spread
A_rep_aon    = tc.assemble_A_spread(store_high, 1, mon, L, tc.rep_assembly(rep_idx, P), P)      # single node
rep_div    = tc.a_diff(A_rep_aon, A_neutral)
spread_div = tc.a_diff(A_spread_aon, A_neutral)
m_neu    = tc.metrics(np.asarray(A_neutral.dot(t_cal)).ravel()[train], v_obs[train])
m_rep    = tc.metrics(np.asarray(A_rep_aon.dot(t_cal)).ravel()[train], v_obs[train])
m_spread = tc.metrics(np.asarray(A_spread_aon.dot(t_cal)).ravel()[train], v_obs[train])
print(f"D3 |A_AON - A_neutral(m x n)| / |A_neutral|:  single-node {rep_div:.0%}  |  N_REP spread {spread_div:.0%}")
print(f"   train fit at Stage-1 t:  m x n AON    {tc.fmt_m(m_neu)}")
print(f"                            single node  {tc.fmt_m(m_rep)}")
print(f"                            N_REP spread {tc.fmt_m(m_spread)}")
print("   -> spreading should sit closer to the m x n neutral representation than a single node does.")

# D4 — training fit at the seed share (the Stage-2 rung of the results ladder).
f_pi0 = np.asarray(A_pi0.dot(t_cal)).ravel()
m_pi0 = tc.metrics(f_pi0[train], v_obs[train])
fl0 = float((tc.PI_LOW0 * A_low).dot(t_cal).sum()); fh0 = float(((1-tc.PI_LOW0) * A_high).dot(t_cal).sum())
print(f"D4 TRAIN at pi0={tc.PI_LOW0}: {tc.fmt_m(m_pi0)}")
print(f"   monitored-flow split at pi0: low {100*fl0/(fl0+fh0):.0f}% / high {100*fh0/(fl0+fh0):.0f}%")


D1 |A_low-A_high|_1/|A_high|_1 = 120%  (distinct -> pi_low identifiable)
   monitored links with a low/high difference: 72/77
   flow lever |f(pi=1)-f(pi=0)|/sum f(pi=0) = 77%
D2 high: median edge overlap among k=3 paths 88%, median cost of path 3 vs path 1 +0.2%
D2 low : median edge overlap among k=3 paths 89%, median cost of path 3 vs path 1 +0.3%
D3 |A_AON - A_neutral(m x n)| / |A_neutral|:  single-node 22%  |  N_REP spread 9%
   train fit at Stage-1 t:  m x n AON    meanGEH 6.35 | GEH<5 54.7% | RMSE 107.4 | R2 0.391 (CoD 0.162) | bias -49% | n=53
                            single node  meanGEH 6.83 | GEH<5 49.1% | RMSE 109.9 | R2 0.375 (CoD 0.122) | bias -51% | n=53
                            N_REP spread meanGEH 6.58 | GEH<5 52.8% | RMSE 108.4 | R2 0.388 (CoD 0.147) | bias -51% | n=53
   -> spreading should sit closer to the m x n neutral representation than a single node does.
D4 TRAIN at pi0=0.81: meanGEH 6.57 | GEH<5 45.3% | RMSE 89.5 | R2 0.557 (CoD 0.418) | bias -33% | n=53

# E — Save, low-stress route behaviour, and report


Save the class matrices and the link-fit CSV, build the per-pair **low-stress route behaviour** table
(`lowstress_route_stats.csv`, consumed by Notebook 07's R2/R3 figures), and write the Stage-2 text
report — this notebook's end-of-run key-results summary.

> **Reconstruction note.** The cell that builds `rs` / `lowstress_route_stats.csv` was missing from the
> handed-over notebook (the report referenced an undefined `rs`, and Notebook 07 reads a file that was
> never written). It is reconstructed below from the cached path stores: for each pair the first-ranked
> low-stress and stress-tolerant paths are compared on length, time and LTS-1 share. Verify the numbers
> against the saved R2/R3 figures before quoting.


In [8]:
# Save the class incidence matrices and the baseline composite.
sparse.save_npz(tc.DATA_PROCESSED / "A_low.npz", A_low)
sparse.save_npz(tc.DATA_PROCESSED / "A_high.npz", A_high)
sparse.save_npz(tc.DATA_PROCESSED / "A_composite_pi0.npz", A_pi0)

# Link-level fit at the seed share, with the per-class flow breakdown, for mapping and tables.
links.assign(f_pi0=f_pi0, f_low_tcal=f_low, f_high_tcal=f_high).to_csv(
    tc.DATA_PROCESSED / "stage2_link_fit.csv", index=False)

# ── Low-stress route behaviour (lowstress_route_stats.csv) ──────────────────────────────────────────
# RECONSTRUCTED CELL (see the markdown note above): the original producer of this table was missing.
# For each O-D pair we take its representative sub-pair's first-ranked path in each class and compare:
#   same_route      - low-stress path identical to the stress-tolerant (time-fastest) path
#   len_low_m/high_m- routed length (m) of each
#   share_lts1_*    - % of each route's length on LTS<=1 links
#   detour_dist/time- % extra distance / time of the low-stress route vs the stress-tolerant route
q = lambda s, p: float(np.nanpercentile(np.asarray(s, float), p))          # percentile helper used below
_rows = []
for pid in range(P):
    ri = int(rep_idx[pid])
    if ri < 0 or not store_high[ri] or not store_low[ri]:
        continue
    p_hi, p_lo = store_high[ri][0], store_low[ri][0]
    T_h, L_h, low1_h = tc.path_stats(Gs_high, p_hi)                        # time(min), length(m), LTS1 length(m)
    T_l, L_l, low1_l = tc.path_stats(Gs_low,  p_lo)
    _rows.append(dict(
        pair_idx=pid, movement=cal.loc[pid, "movement"] if "movement" in cal.columns else "",
        same_route=bool(np.array_equal(p_hi, p_lo)),
        len_high_m=L_h, len_low_m=L_l,
        share_lts1_high=100 * low1_h / max(L_h, 1e-9), share_lts1_low=100 * low1_l / max(L_l, 1e-9),
        detour_dist=100 * (L_l - L_h) / max(L_h, 1e-9), detour_time=100 * (T_l - T_h) / max(T_h, 1e-9)))
rs = pd.DataFrame(_rows)
rs.to_csv(tc.DATA_PROCESSED / "lowstress_route_stats.csv", index=False)

# Stage-2 text report (end-of-run key results).
report = f"""Notebook 05 - Stage 2 report ({tc.RUN_STAMP})
{'='*60}
Two-class Yen k-shortest assignment on N_full, k={tc.K_PATHS} (stored k={tc.K_STORE}), even split.
Low-stress cost: lambda_stress={tc.STRESS_PENALTY} on LTS > {tc.LOW_STRESS_MAX_LTS}.
E_low: {low_e:,}/{n_e:,} directed edges ({100*low_e/n_e:.0f}%); {km[1]:.0f}/{km.sum():.0f} km.

Identifiability |A_low-A_high|/|A_high| = {adiff:.0%}; links with a difference {int((row>1e-9).sum())}/{L}
O-D representation |A_AON - A_neutral|/|A_neutral|:  single-node {rep_div:.0%} | N_REP={tc.N_REP} spread {spread_div:.0%}
  train, Stage-1 t, m x n AON  : {tc.fmt_m(m_neu)}
  train, Stage-1 t, single node: {tc.fmt_m(m_rep)}
  train, Stage-1 t, N_REP spread: {tc.fmt_m(m_spread)}

Low-stress route behaviour (first-ranked path, {len(rs):,} pairs)
  identical to fastest route : {rs.same_route.mean():.0%}
  share on LTS<=1            : median {q(rs.share_lts1_low,50):.0f}%  p10 {q(rs.share_lts1_low,10):.0f}%  p90 {q(rs.share_lts1_low,90):.0f}%
  distance detour (vs fastest route) : median +{q(rs.detour_dist,50):.0f}%  p95 +{q(rs.detour_dist,95):.0f}%
  time detour (vs fastest route)     : median +{q(rs.detour_time,50):.0f}%  p95 +{q(rs.detour_time,95):.0f}%

Held-out stations: not loaded in this notebook.
Outputs: A_low.npz, A_high.npz, A_composite_pi0.npz, stage2_link_fit.csv, lowstress_route_stats.csv
"""
(tc.REPORTS / "05_stage2_report.txt").write_text(report, encoding="utf-8")
print(report)


Notebook 05 - Stage 2 report (20261104_2223)
Two-class Yen k-shortest assignment on N_full, k=3 (stored k=5), even split.
Low-stress cost: lambda_stress=2.4 on LTS > 1.
E_low: 17,905/44,411 directed edges (40%); 797/1944 km.

Identifiability |A_low-A_high|/|A_high| = 120%; links with a difference 72/77
O-D representation |A_AON - A_neutral|/|A_neutral|:  single-node 22% | N_REP=5 spread 9%
  train, Stage-1 t, m x n AON  : meanGEH 6.35 | GEH<5 54.7% | RMSE 107.4 | R2 0.391 (CoD 0.162) | bias -49% | n=53
  train, Stage-1 t, single node: meanGEH 6.83 | GEH<5 49.1% | RMSE 109.9 | R2 0.375 (CoD 0.122) | bias -51% | n=53
  train, Stage-1 t, N_REP spread: meanGEH 6.58 | GEH<5 52.8% | RMSE 108.4 | R2 0.388 (CoD 0.147) | bias -51% | n=53

Low-stress route behaviour (first-ranked path, 1,850 pairs)
  identical to fastest route : 10%
  share on LTS<=1            : median 69%  p10 44%  p90 86%
  distance detour (vs fastest route) : median +8%  p95 +25%
  time detour (vs fastest route)     : median